# Full Benchmark Evaluation (50 Dev + 100 Test)
### Detecting and Repairing Hallucinations in LLM-Generated Answers

This notebook runs the full evaluation benchmark:
- **Dev Set:** 50 records (pipeline + B0)
- **Test Set:** 100 records (pipeline + B0)
- **Index:** FEVER FTS5 Wikipedia lexical index (saved on Google Drive)
- **Model:** Groq API (`qwen/qwen3.8-27b` or `openai/gpt-oss-120b` / `20b`)


In [ ]:
# Step 1: Install dependencies
import subprocess, sys

packages = [
    "pyyaml",
    "rank-bm25",
    "spacy==3.7.5",
    "groq",
    "click",
    "scipy",
    "numpy",
]
subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + packages, check=True)
subprocess.run([sys.executable, "-m", "spacy", "download", "en_core_web_sm"], check=True)
print("Environment dependencies installed successfully.")


In [ ]:
# Step 2: Clone / set up repository
from pathlib import Path
import sys, os, subprocess, json, shutil, zipfile, io

REPO_URL = "https://github.com/harshitap1305/Detecting-and-Repairing-Hallucinations-in-LLM-Generated-Answers.git"
REPO = Path("/content/Detecting-and-Repairing-Hallucinations-in-LLM-Generated-Answers")

if not REPO.exists():
    print(f"Cloning {REPO_URL}...")
    subprocess.run(["git", "clone", REPO_URL, str(REPO)], check=True)
else:
    print("Pulling latest commits...")
    subprocess.run(["git", "-C", str(REPO), "pull"], check=True)

os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

print("Repository ready at:", REPO)


In [ ]:
# Step 3: Mount Google Drive
from google.colab import drive

drive.mount('/content/drive')
DRIVE_ROOT = Path('/content/drive/MyDrive/hallucination_full_benchmark')
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
print("Persistent Google Drive directory:", DRIVE_ROOT)


In [ ]:
# Step 4: Configure Groq API Key & Upload Gold Inputs
from google.colab import files, userdata
import io, zipfile

groq_key = userdata.get("GROQ_API_KEY")
assert groq_key, "GROQ_API_KEY not found in Colab Secrets! Click the key icon on the left sidebar, add GROQ_API_KEY, and toggle Notebook access."
os.environ["GROQ_API_KEY"] = groq_key

GOLD = REPO / "data/gold"
GOLD.mkdir(parents=True, exist_ok=True)

dev_gold = GOLD / "dev.jsonl"
test_gold = GOLD / "test.jsonl"

if not dev_gold.exists() or not test_gold.exists():
    print("Please upload gold_inputs.zip (from your local results/evaluation/gold_inputs.zip):")
    uploads = files.upload()
    for name, data in uploads.items():
        if name.lower().endswith(".zip"):
            with zipfile.ZipFile(io.BytesIO(data)) as zf:
                for member in zf.namelist():
                    extracted = zf.read(member)
                    if member == "splits.json":
                        (REPO / "data/splits.json").write_bytes(extracted)
                    else:
                        (GOLD / Path(member).name).write_bytes(extracted)

assert dev_gold.exists(), "data/gold/dev.jsonl missing!"
assert test_gold.exists(), "data/gold/test.jsonl missing!"
print(f"Dev gold records: {len(dev_gold.read_text().splitlines())}")
print(f"Test gold records: {len(test_gold.read_text().splitlines())}")


In [ ]:
# Step 5: FEVER Wikipedia FTS5 Index
# If the index was already built in Drive from a previous run, reuse it directly!
INDEX = REPO / "indexes/fever_fts.sqlite"
DRIVE_INDEX = DRIVE_ROOT / "fever_fts.sqlite"
# Also check previous pilot folder on Drive if present
ALT_DRIVE_INDEX = Path("/content/drive/MyDrive/hallucination_reviewed_pilot/fever_fts.sqlite")

INDEX.parent.mkdir(parents=True, exist_ok=True)

if not INDEX.exists():
    if DRIVE_INDEX.exists():
        print(f"Reusing existing index from Drive: {DRIVE_INDEX}")
        INDEX.symlink_to(DRIVE_INDEX)
    elif ALT_DRIVE_INDEX.exists():
        print(f"Reusing existing index from pilot Drive: {ALT_DRIVE_INDEX}")
        INDEX.symlink_to(ALT_DRIVE_INDEX)
    else:
        print("Building FEVER FTS5 index (first-time setup)...")
        archive = Path("/content/fever-wiki-pages.zip")
        if not archive.exists():
            import urllib.request
            url = "https://fever.ai/download/fever/wiki-pages.zip"
            print("Downloading FEVER archive:", url)
            urllib.request.urlretrieve(url, str(archive))
        extract_root = Path("/content/fever_wiki")
        extract_root.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(archive) as zipped:
            zipped.extractall(extract_root)
        wiki = extract_root / "wiki-pages"
        subprocess.run([
            sys.executable, "scripts/build_fever_fts.py",
            "--wiki-pages-dir", str(wiki),
            "--output", str(DRIVE_INDEX)
        ], check=True, cwd=REPO)
        INDEX.symlink_to(DRIVE_INDEX)

print("Index verified at:", INDEX, "Size:", f"{INDEX.resolve().stat().st_size / (1024**3):.2f} GB")


In [ ]:
# Step 6: Select & Verify Model
from groq import Groq
client = Groq()
models = [m.id for m in client.models.list().data]
print("Available models on your Groq key:")
for m in sorted(models):
    print(" -", m)

# Default to qwen/qwen3.8-27b or openai/gpt-oss-120b
MODEL = "qwen/qwen3.8-27b" if "qwen/qwen3.8-27b" in models else "openai/gpt-oss-120b"
print(f"\nUsing model: {MODEL}")


In [ ]:
# Step 7: Run Benchmark on Dev Split (50 records)
DEV_OUTPUT = REPO / "results/evaluation/full_benchmark_v1/pipeline_dev"
DEV_OUTPUT.mkdir(parents=True, exist_ok=True)

cmd = [
    sys.executable, "scripts/run_benchmark.py",
    "--config", "configs/evaluation_halueval_colab_full.yaml",
    "--inputs", "data/gold/dev.jsonl",
    "--splits", "data/splits.json",
    "--output-dir", str(DEV_OUTPUT),
    "--task", "repair",
    "--split", "dev",
    "--options-file", "configs/full_benchmark_v1.options.json",
    "--model", MODEL,
    "--resume"
]

print("Executing full benchmark on Dev split...")
subprocess.run(cmd, check=True, cwd=REPO)
print("Dev execution finished!")


In [ ]:
# Step 8: Run Benchmark on Test Split (100 records)
TEST_OUTPUT = REPO / "results/evaluation/full_benchmark_v1/pipeline_test"
TEST_OUTPUT.mkdir(parents=True, exist_ok=True)

cmd = [
    sys.executable, "scripts/run_benchmark.py",
    "--config", "configs/evaluation_halueval_colab_full.yaml",
    "--inputs", "data/gold/test.jsonl",
    "--splits", "data/splits.json",
    "--output-dir", str(TEST_OUTPUT),
    "--task", "repair",
    "--split", "test",
    "--options-file", "configs/full_benchmark_v1.options.json",
    "--model", MODEL,
    "--resume"
]

print("Executing full benchmark on Test split...")
subprocess.run(cmd, check=True, cwd=REPO)
print("Test execution finished!")


In [ ]:
# Step 9: Build Evaluation Progress Reports & Backup to Drive
DEV_REPORT = REPO / "results/evaluation/full_benchmark_v1/report_pipeline_dev"
TEST_REPORT = REPO / "results/evaluation/full_benchmark_v1/report_pipeline_test"

# Dev progress report
subprocess.run([
    sys.executable, "scripts/build_evaluation_report.py",
    "--run-dir", str(DEV_OUTPUT),
    "--gold", "data/gold/dev.jsonl",
    "--output-dir", str(DEV_REPORT),
    "--mode", "progress"
], check=True, cwd=REPO)

# Test progress report
subprocess.run([
    sys.executable, "scripts/build_evaluation_report.py",
    "--run-dir", str(TEST_OUTPUT),
    "--gold", "data/gold/test.jsonl",
    "--output-dir", str(TEST_REPORT),
    "--mode", "progress"
], check=True, cwd=REPO)

# Audit both bundles
subprocess.run([sys.executable, "scripts/audit_baseline.py", "--bundle", str(DEV_REPORT), "--gold", "data/gold/dev.jsonl"], check=True, cwd=REPO)
subprocess.run([sys.executable, "scripts/audit_baseline.py", "--bundle", str(TEST_REPORT), "--gold", "data/gold/test.jsonl"], check=True, cwd=REPO)

# Package all results into Google Drive
ZIP_DEST = DRIVE_ROOT / "full_benchmark_v1_results.zip"
print(f"Creating backup archive at: {ZIP_DEST}")
with zipfile.ZipFile(ZIP_DEST, 'w', zipfile.ZIP_DEFLATED) as zf:
    results_dir = REPO / "results/evaluation/full_benchmark_v1"
    for file_path in results_dir.rglob('*'):
        if file_path.is_file() and not file_path.name.endswith('.sqlite'):
            zf.write(file_path, file_path.relative_to(REPO))

print("Results archive saved successfully to Google Drive:")
print(" ->", ZIP_DEST)
